In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
zip_path = "/content/drive/MyDrive/archive (4).zip"

In [4]:
!unzip -q "/content/drive/MyDrive/archive (4).zip" -d /content/


In [5]:
import os
os.listdir('/content/')

['.config',
 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1',
 'drive',
 'sample_data']

In [6]:
path = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

In [ ]:
os.listdir(path)

['records500',
 'ptbxl_database.csv',
 'scp_statements.csv',
 'RECORDS',
 'SHA256SUMS.txt',
 'LICENSE.txt',
 'records100',
 'example_physionet.py']

ECG ONLY EXPT


In [6]:
# =====================================================
# NEXTBEAT AI – ECG‑ONLY BASELINE (IMPROVED PIPELINE SETTINGS)
# =====================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install wfdb pandas scikit-learn matplotlib seaborn torch torchmetrics tqdm -q
!pip install pandas==2.2.3 -q

import wfdb, pandas as pd, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from tqdm import tqdm
import os, time, warnings, json
from datetime import datetime
warnings.filterwarnings('ignore')

print("✅ Libraries imported")

# ========== CONFIG ==========
data_path = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"
assert os.path.exists(f"{data_path}/ptbxl_database.csv"), "Dataset not found!"

class Config:
    data_path = data_path
    save_folder = "/content/drive/MyDrive/NextBeatAI_Results"
    experiment_name = f"ecg_only_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    sampling_rate = 100
    seq_len = 1000
    num_leads = 12

    # Model
    ecg_cnn_channels = [64, 128, 256]
    ecg_lstm_hidden = 128
    fusion_dim = 128
    num_classes = 5
    dropout = 0.4
    weight_decay = 5e-4
    lr = 0.001
    batch_size = 64
    max_epochs = 80
    patience = 20
    grad_clip = 1.0

    class_names = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def __init__(self):
        os.makedirs(self.save_folder, exist_ok=True)
        self.model_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_best.pth")
        self.results_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_results.json")

cfg = Config()
print(f"Device: {cfg.device}")

# ========== DATA LOADING (standard PTB‑XL labels) ==========
df = pd.read_csv(f"{cfg.data_path}/ptbxl_database.csv", index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(lambda x: eval(x))

def aggregate_diagnostic(y_dict):
    if 'NORM' in y_dict: return 'NORM'
    mi = ['MI','IMI','AMI','PMI','LMI','ILMI','ASMI','ALMI','INJAS','INJAL','IPLMI','IPMI']
    if any(x in y_dict for x in mi): return 'MI'
    sttc = ['STTC','STD','STE','NST_','DIG','LNGQT']
    if any(x in y_dict for x in sttc): return 'STTC'
    cd = ['CD','LAFB','LPFB','IRBBB','CRBBB','CLBBB','ILBBB','LBBB','RBBB','IVCD','WPW','PRC(S)']
    if any(x in y_dict for x in cd): return 'CD'
    hyp = ['HYP','LVH','RVH','LVHYP','RVHYP','SEHYP','LAO/LAE','RAO/RAE']
    if any(x in y_dict for x in hyp): return 'HYP'
    return 'NORM'

df['diagnostic_class'] = df.scp_codes.apply(aggregate_diagnostic)
df = df[df.diagnostic_class.isin(cfg.class_names)].copy()
df['label'] = df.diagnostic_class.map({c:i for i,c in enumerate(cfg.class_names)})

train_df = df[df.strat_fold < 9].copy()
val_df = df[df.strat_fold == 9].copy()
test_df = df[df.strat_fold == 10].copy()

# Clinical features (still needed for the dataset, but the model ignores them)
for split in [train_df, val_df, test_df]:
    split['sex_encoded'] = (split['sex'] == 1).astype(int)
    split['num_diagnoses'] = split.scp_codes.apply(len)

age_scaler = StandardScaler(); num_diag_scaler = StandardScaler()
train_median_age = train_df['age'].median()
train_df['age_scaled'] = age_scaler.fit_transform(train_df[['age']].fillna(train_median_age))
train_df['num_diagnoses_scaled'] = num_diag_scaler.fit_transform(train_df[['num_diagnoses']])
for split in [val_df, test_df]:
    split['age_scaled'] = age_scaler.transform(split[['age']].fillna(train_median_age))
    split['num_diagnoses_scaled'] = num_diag_scaler.transform(split[['num_diagnoses']])

clinical_cols = ['age_scaled', 'sex_encoded', 'num_diagnoses_scaled']

# ========== DATASET (identical to Improved Pipeline) ==========
class PTBXLDataset(Dataset):
    def __init__(self, df, data_path, training=False):
        self.df = df.reset_index(drop=True)
        self.data_path = data_path
        self.n_samples = 1000
        self.training = training
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        record = wfdb.rdrecord(f"{self.data_path}/{row['filename_lr']}",
                               sampfrom=0, sampto=self.n_samples, physical=True)
        signals = record.p_signal.T
        signals = (signals - np.mean(signals, axis=1, keepdims=True)) / (np.std(signals, axis=1, keepdims=True) + 1e-8)
        if signals.shape[1] > self.n_samples: signals = signals[:, :self.n_samples]
        elif signals.shape[1] < self.n_samples:
            signals = np.pad(signals, ((0,0),(0,self.n_samples - signals.shape[1])), mode='constant')
        if self.training:
            shift = np.random.randint(-3, 4)
            if shift > 0: signals = np.pad(signals, ((0,0),(shift,0)), mode='constant')[:, :self.n_samples]
            elif shift < 0: signals = np.pad(signals, ((0,0),(0,-shift)), mode='constant')[:, -shift:]
            signals += 0.02 * np.random.randn(*signals.shape)
        clinical = row[clinical_cols].values.astype(np.float32)
        label = row['label']
        return (torch.tensor(signals, dtype=torch.float32),
                torch.tensor(clinical, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long))

train_ds = PTBXLDataset(train_df, cfg.data_path, training=True)
val_ds   = PTBXLDataset(val_df, cfg.data_path, training=False)
test_ds  = PTBXLDataset(test_df, cfg.data_path, training=False)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

# ========== ECG‑ONLY MODEL ==========
class ECGEncoder(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(12, 64, 5, 2, 2), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64, 128, 5, 2, 2), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(128, 256, 5, 2, 2), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(256, 128, batch_first=True, bidirectional=True)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return torch.cat((h_n[-2], h_n[-1]), dim=1)   # (batch, 256)

class NextBeatAI_ECGOnly(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ecg_encoder = ECGEncoder(dropout=cfg.dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.ecg_encoder.lstm.hidden_size*2, cfg.fusion_dim//2),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fusion_dim//2, cfg.num_classes)
        )

    def forward(self, ecg, clinical=None):
        ecg_feat = self.ecg_encoder(ecg)
        return self.classifier(ecg_feat)

model = NextBeatAI_ECGOnly(cfg).to(cfg.device)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ========== CLASS‑BALANCED LOSS ==========
class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes, device):
        super().__init__()
        effective_num = 1.0 - np.power(0.9999, samples_per_class)
        weights = (1.0 - 0.9999) / np.array(effective_num)
        weights = weights / weights.sum() * num_classes
        self.weights = torch.tensor(weights, dtype=torch.float32).to(device)

    def forward(self, logits, labels):
        return F.cross_entropy(logits, labels, weight=self.weights)

class_counts = train_df['label'].value_counts().sort_index().values
criterion = ClassBalancedLoss(class_counts, cfg.num_classes, cfg.device)

optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# ========== TRAINING FUNCTIONS ==========
def train_epoch():
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for ecg, clin, labels in tqdm(train_loader, desc="Train", leave=False):
        ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
        optimizer.zero_grad()
        logits = model(ecg, clin)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss/len(train_loader), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro')

def evaluate(loader):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0, [], [], []
    with torch.no_grad():
        for ecg, clin, labels in loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            total_loss += criterion(logits, labels).item()
            probs = torch.softmax(logits, 1)
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    all_probs = np.vstack(all_probs)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    try: auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    except: auc = 0.0
    return total_loss/len(loader), acc, f1, auc

# ========== MAIN LOOP ==========
print("\n🚀 Starting ECG‑only training (matching Improved Pipeline)...")
best_val_f1 = 0; patience_counter = 0; start_time = time.time()

for epoch in range(cfg.max_epochs):
    train_loss, train_acc, train_f1 = train_epoch()
    val_loss, val_acc, val_f1, val_auc = evaluate(val_loader)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}: Train Loss={train_loss:.4f} Acc={train_acc:.4f} F1={train_f1:.4f} | "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} AUC={val_auc:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), cfg.model_save_path)
        print(f"   ✅ Best model (F1={val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

print(f"\nTraining finished in {(time.time()-start_time)/60:.1f} min")

# ========== TEST EVALUATION ==========
print("\n📊 TEST SET EVALUATION (ECG‑only)")
if os.path.exists(cfg.model_save_path):
    model.load_state_dict(torch.load(cfg.model_save_path))
    _, test_acc, test_f1, test_auc = evaluate(test_loader)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_f1:.4f}")
    print(f"Test Macro AUROC: {test_auc:.4f}")

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for ecg, clin, labels in test_loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            all_preds.extend(torch.argmax(logits,1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print("\n" + classification_report(all_labels, all_preds, target_names=cfg.class_names, digits=4))

    results = {
        "test_accuracy": float(test_acc) if 'test_acc' in locals() else 0,
        "test_macro_f1": float(test_f1) if 'test_f1' in locals() else 0,
        "test_macro_auroc": float(test_auc) if 'test_auc' in locals() else 0,
        "best_val_f1": float(best_val_f1),
        "model": "ECG-only (CNN-BiLSTM)"
    }
    with open(cfg.results_save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n📁 Results saved to {cfg.results_save_path}")
else:
    print("❌ No saved model found.")

print("\n🎉 ECG‑only baseline complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible

Epoch 01: Train Loss=1.5104 Acc=0.5403 F1=0.2825 | Val Loss=1.4376 Acc=0.5166 F1=0.3245 AUC=0.7127
   ✅ Best model (F1=0.3245)


Epoch 02: Train Loss=1.3456 Acc=0.5644 F1=0.3772 | Val Loss=1.3689 Acc=0.5559 F1=0.3555 AUC=0.7531
   ✅ Best model (F1=0.3555)


Epoch 03: Train Loss=1.2678 Acc=0.5794 F1=0.4189 | Val Loss=1.2630 Acc=0.5463 F1=0.4038 AUC=0.8020
   ✅ Best model (F1=0.4038)


Epoch 04: Train Loss=1.2059 Acc=0.5892 F1=0.4486 | Val Loss=1.2171 Acc=0.6088 F1=0.4312 AUC=0.8235
   ✅ Best model (F1=0.4312)


Epoch 05: Train Loss=1.1457 Acc=0.6239 F1=0.4875 | Val Loss=1.1485 Acc=0.6024 F1=0.4617 AUC=0.8392
   ✅ Best model (F1=0.4617)


... (44 outputs left out)

Epoch 50: Train Loss=0.7205 Acc=0.7427 F1=0.6475 | Val Loss=1.0722 Acc=0.6671 F1=0.5326 AUC=0.8763


Epoch 51: Train Loss=0.7252 Acc=0.7424 F1=0.6474 | Val Loss=1.0503 Acc=0.6772 F1=0.5333 AUC=0.8783


Epoch 52: Train Loss=0.7067 Acc=0.7443 F1=0.6518 | Val Loss=1.0794 Acc=0.6831 F1=0.5347 AUC=0.8790


Epoch 53: Train Loss=0.7004 Acc=0.7469 F1=0.6544 | Val Loss=1.0731 Acc=0.6817 F1=0.5337 AUC=0.8789
⏹️ Early stopping at epoch 53

Training finished in 63.6 min

📊 TEST SET EVALUATION (ECG‑only)
Test Accuracy: 0.6537
Test Macro F1: 0.5184
Test Macro AUROC: 0.8691

              precision    recall  f1-score   support

          CD     0.5604    0.6761    0.6128       247
         HYP     0.2525    0.4322    0.3187       118
          MI     0.8128    0.5497    0.6559       553
        NORM     0.8465    0.7236    0.7802      1212
        STTC     0.1399    0.5616    0.2240        73

    accuracy                         0.6537      2203
   macro avg     0.5224    0.5887    0.5184      2203
weighted avg     0.7508    0.6537    0.6871      2203


📁 Results saved to /content/drive/MyDrive/NextBeatAI_Results/ecg_only_20260504_082559_results.json

🎉 ECG‑only baseline complete!
